# 3.0 - Feature Engineering: What Survived, and What Didn't

**Question this notebook answers:** which engineered features earned a place in the
model, and - just as important - which candidates were tried and dropped, and why?

A feature engineering story that only lists what shipped is incomplete. Two candidates
died during development; showing why is more informative than pretending they never
existed.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from src.data import FEATURE_SET
import src.features as features_module

train = pd.read_parquet(Path('..') / 'data' / 'processed' / 'train.parquet')
print(f'train.parquet: {train.shape}')
print(f'FEATURE_SET (model inputs): {len(FEATURE_SET)} features')


train.parquet: (172988, 89)
FEATURE_SET (model inputs): 79 features


## The 79 features, by family

`FEATURE_SET` (defined once in `src/data.py`, reused by every model in this project) is
not a flat list - it groups into three families:

- **64 "family C" columns** - available at loan origination, survived the leakage
  screen (`2.0`), and are not redundant with another column.
- **10 engineering flags** - `era_pre_2012` plus 9 missingness flags from `2.0`.
- **5 new ratio features** - built row-wise from other family-C columns, described below.

64 + 10 + 5 = 79. `fico_range_high` is a 65th family-C column that exists in
`train.parquet` but was *excluded* from `FEATURE_SET` - see below.

## The 5 features that survived

All five are implemented in `src.features.build_features` - loaded and printed here
directly from the source, not retyped:

In [2]:
import inspect
print(inspect.getsource(features_module.build_features))


def build_features(df):
    """Recebe UM DataFrame, devolve o mesmo com colunas novas. So transformacoes row-wise;
    nao le, nao recebe e nao referencia nenhum outro conjunto.

    fico_mean e bankcard_to_total_limit foram removidas: fico_mean tem correlacao 1.0000
    com fico_range_high (spread low/high e constante, media e apenas translacao linear -
    nao adiciona informacao). bankcard_to_total_limit foi descartada por ter ~30% do treino
    sem valor definido (sentinela em era_pre_2012 mais 0/0 organico) em troca de AUC
    univariada de apenas 0.5393.
    """
    df = df.copy()

    df["installment_to_income"] = df["installment"] / (df["annual_inc"] / 12)
    df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"]
    df["credit_history_months"] = ((df["issue_d"].dt.year - df["earliest_cr_line"].dt.year) * 12
                                    + (df["issue_d"].dt.month - df["earliest_cr_line"].dt.month))
    df["revol_bal_to_income"] = df["revol_bal"] / df["annual_inc"]
   

In [3]:
new_features = ['installment_to_income', 'loan_to_income', 'credit_history_months',
                'revol_bal_to_income', 'open_acc_ratio']
train[new_features].describe().T


,count,mean,std,min,25%,50%,75%,max
installment_to_income,172988.0,0.078444,0.042402,0.000289,0.045598,0.072305,0.105997,0.320262
loan_to_income,172988.0,0.194426,0.103569,0.000789,0.114286,0.180000,0.263158,0.830000
credit_history_months,172988.0,179.386853,85.529539,36.000000,122.000000,163.000000,221.000000,785.000000
revol_bal_to_income,172988.0,0.228868,0.180945,0.000000,0.108721,0.191887,0.305725,6.086388
open_acc_ratio,172988.0,0.491985,0.177495,0.000000,0.361702,0.470588,0.600000,1.750000


Each is a ratio or a derived duration, not a raw magnitude - the working hypothesis
being that *relative* burden (installment against income, loan size against income,
utilization against account count) generalizes better across the wide income range in
this population than the raw dollar amounts alone. `installment_to_income` and `dti`
turn out to rank #2 and #5 by SHAP importance in the final model (`6.0`) - the hypothesis
held up.

## The 2 features that died

Neither made it into `train.parquet` at all - they are documented here from the
`build_features` docstring above and from `docs/cleaning_decisions.md`, not recomputed:

1. **`fico_mean`** (average of `fico_range_low` and `fico_range_high`) - dropped for
   **redundancy**, not weakness. Lending Club reports FICO as a fixed-width band (the
   high bound is always exactly 4 points above the low bound in this data), so the mean
   is a pure linear transformation of `fico_range_low` alone and adds no information.
2. **`bankcard_to_total_limit`** (a bankcard-vs-total-credit ratio) - dropped for
   **missingness plus weak signal**: ~30% of the training set had it undefined (0/0 from
   borrowers with no bankcard, compounded by the same sentinel pattern from `2.0`), in
   exchange for a univariate AUC of only 0.5393 - barely better than chance, not worth the
   missingness it would introduce.

Dropping a feature that would have "sounded" reasonable in a report is the point, not a
gap: it shows the ratio features that did ship were screened against alternatives, not
the only two ideas tried.

In [4]:
corr = train[['fico_range_low', 'fico_range_high']].corr().iloc[0, 1]
spread = (train['fico_range_high'] - train['fico_range_low'])
print(f'Correlation(fico_range_low, fico_range_high) = {corr:.6f}')
print(f'fico_range_high - fico_range_low: unique values = {sorted(spread.unique())}')


Correlation(fico_range_low, fico_range_high) = 1.000000
fico_range_high - fico_range_low: unique values = [np.float64(4.0), np.float64(5.0)]


The correlation is (numerically) exactly 1.0, and the spread between the two bounds
takes on a single fixed value across the entire training set - direct, live confirmation
of the redundancy claim above, computed on the actual processed data rather than taken on
faith.

## Takeaway

79 model features: 64 origination-time columns that passed the leakage and redundancy
screens, 10 flags that disambiguate sentinel values from real signal (`2.0`), and 5
engineered ratios chosen because relative burden was hypothesized to generalize better
than raw magnitudes. Two more candidates were tried and rejected on quantitative grounds
(perfect collinearity; missingness with negligible payoff) - the same discipline applied
to what shipped was applied to what didn't.

**Next:** `4.0-modelling.ipynb` - the walk-forward temporal validation scheme and the two
finalist models it selected between.